# AMPR Phase 3 — Evaluate MF Branch (Diamond + Calibration + Metrics)

Run on **Kaggle CPU** (~30 minutes) after training completes.

Steps:
1. Install Diamond aligner
2. Build Diamond DB from train FASTA; search test against it
3. Load val probs from checkpoint; tune Diamond ensemble alpha on valid set
4. Calibrate threshold on valid set
5. Compute final metrics on test and test_LT_xx splits
6. Print comparison table vs HEAL S3.2 / AMPR v1

In [ ]:
# Install Diamond
!apt-get install -y diamond-aligner 2>/dev/null || \
  (wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.9/diamond-linux64.tar.gz \
   && tar xzf diamond-linux64.tar.gz && mv diamond /usr/local/bin/)
!diamond --version

In [ ]:
import os
os.chdir('/kaggle/working/datn')

# Build Diamond DB and search
!bash scripts/run_diamond.sh mf 2>&1 | tail -20

In [ ]:
import numpy as np
import torch
import json
from pathlib import Path
from torch.utils.data import DataLoader

from ampr.data.dataset import AMPRDatasetV3, collate_variable_length
from ampr.models.ampr import AMPRModelV3
from ampr.evaluation.dag_inference import propagate_scores_upward
from ampr.evaluation.diamond_ensemble import compute_diamond_scores, ensemble_scores, tune_alpha
from ampr.evaluation.threshold_calibration import calibrate_and_save
from ampr.evaluation.metrics import compute_fmax, compute_all_metrics

cfg = json.loads(Path('configs/mf_v3.yaml').read_text())
# (load config via yaml)
import yaml
with open('configs/mf_v3.yaml') as f:
    cfg = yaml.safe_load(f)

labels_all = np.load(cfg['data']['labels'])
splits = json.loads(Path(cfg['data']['splits']).read_text())
order = json.loads(Path(cfg['data']['protein_order']).read_text())
if isinstance(order, dict):
    order = [k for k, _ in sorted(order.items(), key=lambda kv: kv[1])]
prot2idx = {p: i for i, p in enumerate(order)}
print(f'Labels shape: {labels_all.shape}')

In [ ]:
# Load model and collect val + test probs
device = 'cpu'

def collect_probs(split_key):
    ds = AMPRDatasetV3(
        esm2_h5=cfg['data']['esm2_h5'],
        ppi_emb=cfg['data']['ppi_emb'],
        ppi_mask=cfg['data']['ppi_mask'],
        cmap_h5=cfg['data']['cmap_h5'],
        labels=cfg['data']['labels'],
        dag_matrix=cfg['data']['dag_matrix'],
        go_emb=cfg['data']['go_emb'],
        splits=cfg['data']['splits'],
        protein_order=cfg['data']['protein_order'],
        branch=cfg['branch'], split=split_key,
        max_len=cfg['training']['max_seq_len'],
    )
    loader = DataLoader(ds, batch_size=32, collate_fn=collate_variable_length)
    probs_list, labels_list, ids_list = [], [], []
    model.eval()
    with torch.no_grad():
        for b in loader:
            logits = model(b, go_emb=go_emb)
            probs_list.append(torch.sigmoid(logits).numpy())
            labels_list.append(b['labels'].numpy())
            ids_list.extend(b['prot_id'])
    return np.concatenate(probs_list), np.concatenate(labels_list), ids_list

# Build model + load checkpoint
mc = cfg['model']
sc, gc, pc, fc = mc['seq'], mc['gnn'], mc['ppi'], mc['fusion']
go_emb_np = np.load(cfg['data']['go_emb'])
go_emb = torch.from_numpy(go_emb_np)

model = AMPRModelV3(
    n_terms=cfg['n_terms'],
    seq_dim=sc['d_model'], seq_n_heads=sc['n_heads'], seq_n_layers=sc['n_transformer_layers'],
    gnn_node_dim=gc['node_dim'], gnn_n_layers=gc['n_layers'],
    ppi_dim=pc['in_dim'], d_hidden=mc['d_hidden'],
    fusion_n_heads=fc['n_heads'], fusion_n_layers=fc['n_layers'],
    classifier=mc['classifier'], go_emb_dim=go_emb.shape[1],
    cmap_threshold=gc['cmap_threshold'],
)
ckpt = torch.load('checkpoints/mf_v3/best.pt', map_location='cpu')
model.load_state_dict(ckpt['model'])
print(f'Loaded checkpoint (epoch={ckpt["epoch"]}, fmax_dag={ckpt["fmax_dag"]:.4f})')

val_probs, val_labels, val_ids = collect_probs('valid')
test_probs, test_labels, test_ids = collect_probs('test')
print(f'Val: {val_probs.shape}, Test: {test_probs.shape}')

In [ ]:
# Diamond ensemble: compute scores and tune alpha
dag_matrix = np.load(cfg['data']['dag_matrix'])
train_ids = splits['train']
train_labels_arr = np.stack([labels_all[prot2idx[p]] for p in train_ids if p in prot2idx])
train_order_map = {p: i for i, p in enumerate(train_ids) if p in prot2idx}

n_terms = cfg['n_terms']
val_diamond = compute_diamond_scores(
    'data/diamond/diamond_mf_valid.tsv', train_labels_arr, train_order_map, val_ids, n_terms)
test_diamond = compute_diamond_scores(
    'data/diamond/diamond_mf_test.tsv', train_labels_arr, train_order_map, test_ids, n_terms)

best_alpha = tune_alpha(val_probs, val_diamond, val_labels, dag_matrix)
print(f'Best ensemble alpha: {best_alpha:.2f}')

val_ens = ensemble_scores(val_probs, val_diamond, best_alpha)
test_ens = ensemble_scores(test_probs, test_diamond, best_alpha)

In [ ]:
# DAG propagation + threshold calibration
val_dag = propagate_scores_upward(val_ens, dag_matrix)
test_dag = propagate_scores_upward(test_ens, dag_matrix)

from pathlib import Path
Path('checkpoints/mf_v3').mkdir(parents=True, exist_ok=True)
cal = calibrate_and_save(
    val_dag, val_labels, branch='MF',
    output_path='checkpoints/mf_v3/threshold.json'
)
thresh = cal['threshold']
print(f'Calibrated threshold: {thresh:.3f}, val Fmax: {cal["val_fmax"]:.4f}')

In [ ]:
# Final metrics table
import pandas as pd

rows = []
test_split_keys = ['test', 'test_LT_30', 'test_LT_40', 'test_LT_50', 'test_LT_70', 'test_LT_95']

for split_key in test_split_keys:
    if split_key not in splits:
        continue
    s_ids = splits[split_key]
    idx = [test_ids.index(p) for p in s_ids if p in test_ids]
    if not idx:
        continue
    p_sub = test_dag[idx]
    l_sub = test_labels[idx]
    fmax, t = compute_fmax(l_sub, p_sub)
    rows.append({'split': split_key, 'Fmax': fmax, 'n': len(idx)})

df = pd.DataFrame(rows)
print('\nAMPR v3 MF results:')
print(df.to_string(index=False))

# HEAL DeepFRI reference (S3.2)
HEAL_REF = {'test': 0.619, 'test_LT_30': 0.421, 'test_LT_40': 0.480,
            'test_LT_50': 0.506, 'test_LT_70': 0.555, 'test_LT_95': 0.599}
df['HEAL_DeepFRI'] = df['split'].map(HEAL_REF)
df['vs_HEAL'] = df['Fmax'] - df['HEAL_DeepFRI']
print('\nComparison vs HEAL DeepFRI (S3.2):')
print(df[['split', 'Fmax', 'HEAL_DeepFRI', 'vs_HEAL', 'n']].to_string(index=False))

df.to_csv('results/mf_v3_eval.tsv', sep='\t', index=False)
print('\nSaved to results/mf_v3_eval.tsv')